# Deduplication — perceptual hashing

Removes near-duplicate images using `imagededup` (pHash).

**Why perceptual rather than exact hashing:** the same viral meme is re-encoded, re-compressed,
watermarked and cropped as it spreads. MD5/SHA treat these as entirely different files. Perceptual
hashing produces similar hashes for perceptually similar images, so near-duplicates collapse.

**Why this is a correctness requirement, not housekeeping:** the same memes appear across Reddit,
4chan and every image search index. Duplicates spanning a train/test boundary leak the test set —
the model scores well by memorising, and reported accuracy becomes meaningless.

**Known limitation:** the pHash distance threshold and the number of pairs merged were not logged,
so this step is not exactly reproducible from the notebook alone.


In [ ]:
pip install imagededup opencv-python matplotlib


In [ ]:
import os
from google.colab import drive

In [ ]:
# Step 1: Mount Google Drive
drive.mount('/content/drive')

In [ ]:
dataset_path = "/content/drive/MyDrive/LGBTQ Memes/Combined_1500"

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
from imagededup.methods import PHash

In [ ]:
# Initialize the PHash model
phash = PHash()

# Generate hashes for all images
encodings = phash.encode_images(image_dir=dataset_path)

# Find duplicates with a similarity threshold
duplicates = phash.find_duplicates(encoding_map=encodings, max_distance_threshold=5)

In [ ]:
# Store duplicate image pairs
duplicate_pairs = []

# Process and store duplicate pairs
for original, dup_list in duplicates.items():
    for duplicate in dup_list:
        original_path = os.path.join(dataset_path, original)
        duplicate_path = os.path.join(dataset_path, duplicate)
        duplicate_pairs.append((original_path, duplicate_path))

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# Function to display duplicates side by side
def display_duplicates(pairs):
    for original, duplicate in pairs:
        try:
            img1 = cv2.imread(original)
            img2 = cv2.imread(duplicate)

            if img1 is None or img2 is None:
                print(f"Could not load images: {os.path.basename(original)}, {os.path.basename(duplicate)}")
                continue

            img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
            img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

            # Display images side by side
            fig, ax = plt.subplots(1, 2, figsize=(10, 5))
            ax[0].imshow(img1)
            ax[0].set_title(f"Original\n{os.path.basename(original)}")
            ax[0].axis('off')

            ax[1].imshow(img2)
            ax[1].set_title(f"Duplicate\n{os.path.basename(duplicate)}")
            ax[1].axis('off')

            plt.show()

        except Exception as e:
            print(f"Error displaying images: {e}")

# Print duplicate pairs with filenames only
print("\n=== Duplicate Images Found ===")
for orig, dup in duplicate_pairs:
    print(f"Original: {os.path.basename(orig)}  ||  Duplicate: {os.path.basename(dup)}")

# Display duplicates
display_duplicates(duplicate_pairs)

print("\nDuplicate detection complete! Review images before manual deletion.")


In [ ]:
unique_folder = os.path.join(dataset_path, "unique_images")

# Create folder for unique images (copy everything first)
os.makedirs(unique_folder, exist_ok=True)

In [ ]:
import shutil

In [ ]:
# Copy all images from original folder to the new folder
for image in os.listdir(dataset_path):
    src_path = os.path.join(dataset_path, image)
    dest_path = os.path.join(unique_folder, image)

    if os.path.isfile(src_path):  # Ensure it's a file
        shutil.copy2(src_path, dest_path)

In [ ]:
# Initialize PHash model
phash = PHash()

# Generate hashes for all images in the new folder
encodings = phash.encode_images(image_dir=unique_folder)

# Find duplicates
duplicates = phash.find_duplicates(encoding_map=encodings, max_distance_threshold=5)

# Track images to delete (keep the first occurrence)
deleted_files = set()

for original, dup_list in duplicates.items():
    for duplicate in dup_list:
        duplicate_path = os.path.join(unique_folder, duplicate)

        if os.path.exists(duplicate_path) and duplicate not in deleted_files:
            os.remove(duplicate_path)  # Delete duplicate
            deleted_files.add(duplicate)  # Mark as deleted

print(f"✅ Duplicate images removed from: {unique_folder}")
print(f"📁 Unique images are now stored in: {unique_folder}")

In [ ]:
# Count the number of files in the unique_folder
num_files = 0
if os.path.exists(unique_folder):
    num_files = len([f for f in os.listdir(unique_folder) if os.path.isfile(os.path.join(unique_folder, f))])
    print(f"Number of files in {unique_folder}: {num_files}")
else:
    print(f"Error: The folder '{unique_folder}' does not exist.")


In [ ]:
# Generate hashes for all images in the new folder
encodings = phash.encode_images(image_dir=unique_folder)

# Find duplicates
duplicates = phash.find_duplicates(encoding_map=encodings, max_distance_threshold=5)

In [ ]:
print(duplicates)

In [ ]:
# Count duplicates
total_duplicates = sum(len(dup_list) for dup_list in duplicates.values())

# Function to display duplicates side by side
def display_duplicates(pairs):
    for original, duplicate in pairs:
        try:
            img1 = cv2.imread(os.path.join(unique_folder, original))
            img2 = cv2.imread(os.path.join(unique_folder, duplicate))

            if img1 is None or img2 is None:
                print(f"Could not load images: {original}, {duplicate}")
                continue

            img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
            img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

            # Display images side by side
            fig, ax = plt.subplots(1, 2, figsize=(10, 5))
            ax[0].imshow(img1)
            ax[0].set_title(f"Original\n{original}")
            ax[0].axis('off')

            ax[1].imshow(img2)
            ax[1].set_title(f"Duplicate\n{duplicate}")
            ax[1].axis('off')

            plt.show()

        except Exception as e:
            print(f"Error displaying images: {e}")

# Print duplicate pairs
if total_duplicates > 0:
    print(f"\n=== Found {total_duplicates} Duplicate Images ===")
    duplicate_pairs = []

    for orig, dup_list in duplicates.items():
        for dup in dup_list:
            print(f"Original: {orig}  ||  Duplicate: {dup}")
            duplicate_pairs.append((orig, dup))

    # Display duplicates
    display_duplicates(duplicate_pairs)
else:
    print("\n✅ No duplicates found!")

print("\nDuplicate detection complete! Review images before manual deletion.")